# cv_r1 話者評価【Colab・要 GPU・自己完結版】

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/slp-hu/Style-Bert-VITS2/blob/layer-b-cadence-seq/colab/cv_r1_eval_speaker.ipynb)

学習完走後に、多人数話者合成の **弾き分け（speaker discriminability）** と **自然性（UTMOS）** を評価し、
**virtual 話者**（emb_g 補間・平均）を試聴する。**仮名化（cadence 変換）はしない**
（既定 `WEIGHT=0.0` = cadence 非注入、emb_g のみで弾き分け）。

学習セッションが生きている必要はない。**§0 系がゼロから環境を再構築**する:
torch（GPU 自動判定: 標準 = 2.3.1 / **Blackwell** = 2.11+cu128 入替 → **要ランタイム再起動** →
§0-1 から再実行）→ shim → Zenodo データをローカル `/content` に展開 → Drive の `model_assets`
（学習済み ckpt / style_vectors）を接続 → `xvec_extract.py` を配置。

**前提**
- setup 済み（Drive に fork clone）・**学習完走済み**（Drive の `model_assets/` に `*_e10_s*.safetensors`）
- **GPU ランタイム**

**評価軸**
- 弾き分け: 合成×原音の x-vector cos 類似度行列／top-1 話者識別率（chance=1/N）／話者内・話者間分布
- 自然性: UTMOS（合成 vs 原音）
- virtual: 話者A↔B の emb_g 補間 α=0..1、複数話者の平均（重心）

**役割分担（Blackwell でも動く構成）**
- 合成 / UTMOS = torch（GPU）。UTMOS は torch.hub `tarepan/SpeechMOS:v1.2.0`（初回 392MB DL）。
- x-vector（`CV_SPKREC.keras`）= TF/keras を **CPU 固定・別プロセス**（`xvec_extract.py`）で実行。
  Colab の TF ビルドは Blackwell 非対応で、GPU を掴むと `CUDA_ERROR_INVALID_HANDLE` で落ちるため。
  モデル / LDA は公開 GitHub（`sp-au-mu-nl/SpeechComm` chap10.zip）から自動取得。

**参考（実測 2026-07, N=10 話者 × M=3 発話, weight=0）**: top-1 話者識別率 76.7%（chance 10%）／
cos 話者内 0.434・話者間 0.099（差 0.335）／UTMOS 原音 2.369・合成 2.398（ギャップ −0.029 = 実質同等）。
平均話者（重心）が最も自然に聞こえる。


In [ ]:
# ===== §0-1 マウント・前提チェック【毎回・ランタイム再起動後も最初に実行】=====
# --- Colab カーネル preload 対策 ---------------------------------------------
# Colab はカーネルを主要ライブラリ preload 済みスナップショットから起動するため、
# ディスクの numpy を入れ替えても、カーネルには旧版が sys.modules に残り続ける
# （再起動でも直らない）。版がズレていたら preload を破棄してディスク版を読み直させる。
import sys, subprocess

def _purge_stale_numpy():
    disk = subprocess.run([sys.executable, "-c", "import numpy; print(numpy.__version__)"],
                          capture_output=True, text=True).stdout.strip()
    k = sys.modules.get("numpy")
    if k is None or not disk or getattr(k, "__version__", "") == disk:
        return
    victims = [m for m in list(sys.modules) if m.split(".")[0] in
               ("numpy", "pandas", "scipy", "matplotlib", "mpl_toolkits", "sklearn", "pyarrow", "PIL")]
    for m in victims:
        del sys.modules[m]
    import numpy as _np
    assert _np.__version__ == disk, f"★numpy 差し替え失敗: {_np.__version__}（ランタイム再起動でやり直す）"
    import numpy.random   # サブパッケージまで健全なことを確認
    print(f"カーネル preload の numpy {getattr(k, '__version__', '?')} 系 {len(victims)} モジュールを破棄 → {disk} に差し替え")

_purge_stale_numpy()
# -------------------------------------------------------------------------------
from google.colab import drive
from pathlib import Path
import os, subprocess

DRIVE_BASE = Path("/content/drive/MyDrive/Style-Bert-VITS2")   # setup と同じ値（clone の置き場所）

def _drive_alive():
    """マウントの生死確認（stale mount だと listdir が OSError: Transport endpoint is not connected）"""
    try:
        next(iter(os.listdir("/content/drive/MyDrive")), None)
        return True
    except OSError:
        return False

drive.mount("/content/drive")
if not _drive_alive():
    print("★Drive マウントが切れている（Transport endpoint is not connected）→ 強制再マウント")
    subprocess.run(["fusermount", "-u", "/content/drive"], capture_output=True)
    drive.mount("/content/drive", force_remount=True)
    assert _drive_alive(), "★再マウント失敗 → ランタイム再起動してやり直す"
assert DRIVE_BASE.exists(), f"★Drive に fork clone が無い: {DRIVE_BASE}（先に cv_r1_train_setup_colab.ipynb を実行）"

# --- fork の更新を Drive clone に自動反映（ベストエフォート。失敗しても続行）---
# バッジで開くノートは常に GitHub の最新だが、コードは Drive clone のスナップショット。
# ここで揃えないと「新しいノート × 古いコード」の不整合が起きる。意図的に版を固定したい場合は False。
AUTO_PULL = True
if AUTO_PULL:
    try:
        _p = subprocess.run(["git", "-C", str(DRIVE_BASE), "pull", "--ff-only"],
                            capture_output=True, text=True, timeout=180)
        if _p.returncode == 0:
            print("git pull:", (_p.stdout.strip().splitlines() or ["?"])[-1])
        else:
            print("★git pull 失敗（そのまま続行。clone の手元変更やネットワークを確認）:",
                  (_p.stderr or "").strip()[-200:])
    except Exception as e:
        print("★git pull 例外（そのまま続行）:", e)
_h = subprocess.run(["git", "-C", str(DRIVE_BASE), "rev-parse", "--short", "HEAD"],
                    capture_output=True, text=True).stdout.strip()
print("clone commit:", _h)

os.chdir(DRIVE_BASE); print("cwd:", Path.cwd())   # 依存インストールは requirements.txt をここから読む（後段でローカルへ移る）

br = subprocess.run(["git","rev-parse","--abbrev-ref","HEAD"], capture_output=True, text=True).stdout.strip()
assert br == "layer-b-cadence-seq", f"★branch が違う: {br} → git checkout layer-b-cadence-seq"
print("branch:", br)

g = subprocess.run(["nvidia-smi","-L"], capture_output=True, text=True)
assert g.returncode == 0 and "GPU" in g.stdout, "★GPU ランタイムでない → [ランタイム]→[ランタイムのタイプを変更]→GPU"
print(g.stdout.strip())


In [ ]:
# ===== §0-2 GPU 判定と依存インストール【毎セッション実行・再起動後も再実行（冪等）】=====
# GPU の compute capability で経路を自動分岐する:
#   ・sm_90 以下（T4/L4/A100 等）: requirements の pin どおり torch 2.3.1(cu121)。再起動不要。
#   ・sm_100 以上（Blackwell 系: RTX PRO 6000 = sm_120 等）: torch 2.3.1 は sm_90 までで非対応
#     （GPU forward で落ちる）→ torch 2.11.0+cu128 へ入替（07a 方式）。★入替後はランタイム再起動が必須。
# 再起動後にこのセルを再実行すると、入替をスキップして検証だけ行う。
import subprocess, sys, re, os
from pathlib import Path

def _run(cmd, stream=False):
    print("$", " ".join(cmd))
    if stream:   # 進捗をそのまま流す（★torch の数GB DL は -q だと無言=フリーズと誤認するため）
        p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        for line in p.stdout: print(line, end="")
        p.wait(); assert p.returncode == 0, f"失敗: {cmd}"
    else:
        r = subprocess.run(cmd, capture_output=True, text=True)
        print((r.stdout or "")[-1200:] or "(quiet)")
        if r.returncode != 0:
            print(r.stderr[-4000:]); raise SystemExit(f"失敗: {cmd}")

# --- GPU capability（torch を import せず nvidia-smi で判定するのが肝）---
cap = subprocess.run(["nvidia-smi","--query-gpu=compute_cap","--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip().splitlines()
assert cap and cap[0], "★GPU が見えない（GPU ランタイムか確認）"
CAP = float(cap[0]); IS_BLACKWELL = CAP >= 10.0
print(f"compute capability: {cap[0]} (sm_{int(CAP*10)}) → 経路: "
      + ("Blackwell（torch 2.11+cu128 入替）" if IS_BLACKWELL else "標準（torch 2.3.1 のまま）"))

# --- 現在の torch を subprocess で確認（in-kernel import しない = 再起動不要性を保つ）---
q = subprocess.run([sys.executable,"-c","import torch;print(torch.__version__)"], capture_output=True, text=True)
TORCH_NOW = q.stdout.strip()
print("現在の torch:", TORCH_NOW or "(未導入)")
ALREADY_SWAPPED = IS_BLACKWELL and TORCH_NOW.startswith("2.11.")

if ALREADY_SWAPPED:
    print("→ 再起動後の再実行と判断: torch 入替済みのため install をスキップ")
else:
    # (1) requirements から faster-whisper を除外して install（07a 方式）。
    #     faster-whisper==0.10.1 が av==10.* をソースビルドしようとし Py3.12 で失敗するため。
    #     文字起こし用途で cv_r1（bert_gen/style_gen/train/eval）には不要。
    req = Path("requirements.txt"); req_train = Path("requirements_no_whisper.txt")
    _drop = re.compile(r"^(faster-whisper|av)==")   # ビルドで詰まるパッケージが増えたらここに追加（例: stable_ts）
    kept = [l for l in req.read_text(encoding="utf-8").splitlines() if not _drop.match(l.strip())]
    req_train.write_text("\n".join(kept) + "\n", encoding="utf-8")
    _run([sys.executable,"-m","pip","install","-q","-r",str(req_train)])

    # (2) Blackwell のみ: torch 2.11.0+cu128 へ入替（実績のある手順をそのまま踏む）
    if IS_BLACKWELL:
        _run([sys.executable,"-m","pip","uninstall","-y",
              "torch","torchaudio","torchvision","torchcodec","torchao","torchtune","torchdata"])
        _run([sys.executable,"-m","pip","install","torch==2.11.0","torchaudio==2.11.0",
              "--index-url","https://download.pytorch.org/whl/cu128"], stream=True)   # ★-q 禁止
        _run([sys.executable,"-m","pip","install","-q","soundfile"])
        _run([sys.executable,"-m","pip","uninstall","-y",
              "torchcodec","torchvision","torchao","torchtune","torchdata"])

    # (3) HF スタック固定（transformers 未ピン → Colab 既定の transformers 5.x が torch>=2.4 を要求して
    #     torch を無効化し、bert_gen が "AutoModelForMaskedLM requires PyTorch" で落ちる問題の恒久対策）
    _run([sys.executable,"-m","pip","install","-q",
          "transformers==4.41.2","huggingface_hub==0.23.5","tokenizers<0.20",
          "pytorch-lightning==2.2.5","torchmetrics<1.5","pyannote.audio==3.1.1",
          "scipy==1.13.1","numpy==1.26.4"])

# --- numpy 健全性チェック（混在インストールの検出と自動修復）---
# pip のダウングレードで旧 2.x の compiled .so（numpy/random/mtrand 等）が残ると、
# コアは import できるのにサブパッケージ初期化で "numpy.dtype size changed" になる。
_h = subprocess.run([sys.executable,"-c","import numpy.random, numpy; print(numpy.__version__)"],
                    capture_output=True, text=True)
if _h.returncode != 0:
    print("★numpy が混在状態（サブパッケージ初期化に失敗）→ 1.26.4 を強制再インストールで修復")
    print("  症状:", (_h.stderr or "").strip().splitlines()[-1] if _h.stderr else "?")
    _run([sys.executable,"-m","pip","install","-q","--force-reinstall","--no-deps",
          "--no-cache-dir","numpy==1.26.4"])
    _h = subprocess.run([sys.executable,"-c","import numpy.random, numpy; print(numpy.__version__)"],
                        capture_output=True, text=True)
    assert _h.returncode == 0, "★numpy を修復できない:\n" + (_h.stderr or "")[-600:]
print("numpy 健全性 OK:", _h.stdout.strip())

# --- 検証（別プロセス。transformers から torch が見えているかまで確認）---
v = subprocess.run([sys.executable,"-c",
    "import torch;from transformers.utils import is_torch_available;"
    "print('torch',torch.__version__,'| cuda',torch.cuda.is_available(),"
    "'| transformers_sees_torch',is_torch_available())"], capture_output=True, text=True)
print(v.stdout.strip() or v.stderr[-800:])
assert "transformers_sees_torch True" in v.stdout, "★transformers が torch を認識していない → このセルをやり直す"
if (not IS_BLACKWELL) or ALREADY_SWAPPED:
    assert "cuda True" in v.stdout, "★CUDA が使えない（GPU ランタイム / torch ビルドを確認）"

# --- カーネル整合性チェック: カーネルに import 済みの numpy とディスクの numpy が食い違うと、
#     このノートの in-kernel import（torch/numba 等）が "numpy.dtype size changed" で落ちる。
#     （Colab 素の numpy 2.x をカーネルが先に読み込み、上の install が 1.26.4 へ下げた場合に起きる）
_disk_np = subprocess.run([sys.executable,"-c","import numpy;print(numpy.__version__)"],
                          capture_output=True, text=True).stdout.strip()
_knp = sys.modules.get("numpy")
NUMPY_MISMATCH = _knp is not None and getattr(_knp, "__version__", "") != _disk_np

if NUMPY_MISMATCH:
    # Colab のカーネル preload（snapshot）由来のズレは再起動では直らない → その場で差し替える
    print(f"カーネルの numpy {_knp.__version__} ≠ ディスク {_disk_np} → preload を破棄して差し替え")
    for _m in [x for x in list(sys.modules) if x.split(".")[0] in
               ("numpy", "pandas", "scipy", "matplotlib", "mpl_toolkits", "sklearn", "pyarrow", "PIL")]:
        del sys.modules[_m]
    import numpy as _np_chk
    assert _np_chk.__version__ == _disk_np, f"★差し替え失敗: {_np_chk.__version__}（ランタイム再起動でやり直す）"
    import numpy.random
    print("カーネル numpy:", _np_chk.__version__, "（再起動不要）")

if IS_BLACKWELL and not ALREADY_SWAPPED:
    print()
    print("=" * 70)
    print("★torch を入れ替えた → ここで【ランタイム再起動】が必須★")
    print("  [ランタイム] → [セッションを再起動] のあと、先頭セルから順に再実行してから先へ進む。")
    print("  （再実行時は install が即完了し、この警告は出なくなる）")
    print("=" * 70)
else:
    print("環境 OK → 次のセルへ")


In [ ]:
# ===== §0-3 torchaudio / torch.load shim【Blackwell 経路のみ実体化・毎セッション実行】=====
# torch 2.11 系では torchaudio.set_audio_backend が削除され、torchaudio.load も torchcodec 経由で壊れる。
# pyannote.audio が import 時に set_audio_backend を呼ぶため、shim なしでは style_gen / 合成が落ちる。
# `!python` で走る bert_gen / style_gen / train は【別プロセス】なので、カーネル内 monkeypatch では効かない
# → sitecustomize.py + PYTHONPATH で全 Python プロセスに注入する（ここが肝）。
import os, subprocess, sys
from pathlib import Path

if not IS_BLACKWELL:
    print("標準経路（torch 2.3.1）: shim 不要 → スキップ")
else:
    compat = Path("/content/_compat"); compat.mkdir(exist_ok=True)
    shim = compat / "sitecustomize.py"
    SHIM_SRC = '# sitecustomize: torch 2.11 環境の互換 shim（cv_r1 公開ノート用）\n# 1) torch.load の weights_only 既定を False に戻す（旧 ckpt / torch.hub モデルの読込互換）\n# 2) torchaudio.set_audio_backend / get_audio_backend を復活（pyannote.audio の import 時呼び出し対策）\n# 3) torchaudio.load / info を soundfile 実装に置換（torchcodec 経由の破綻を回避）\ntry:\n    import torch\n    _orig_torch_load = torch.load\n    def _patched_torch_load(*args, **kwargs):\n        kwargs.setdefault("weights_only", False)\n        return _orig_torch_load(*args, **kwargs)\n    torch.load = _patched_torch_load\nexcept Exception:\n    pass\n\ntry:\n    import torchaudio\n\n    def _noop_set_backend(*args, **kwargs):\n        return None\n    def _get_backend(*args, **kwargs):\n        return "soundfile"\n    torchaudio.set_audio_backend = _noop_set_backend\n    torchaudio.get_audio_backend = _get_backend\n\n    def _sf_load(filepath, frame_offset=0, num_frames=-1, normalize=True,\n                 channels_first=True, format=None, buffer_size=4096, backend=None):\n        import soundfile as sf\n        import torch as _torch\n        frames = int(num_frames) if int(num_frames) > 0 else -1\n        data, sr = sf.read(str(filepath), start=int(frame_offset), frames=frames,\n                           dtype="float32", always_2d=True)\n        wav = _torch.from_numpy(data.T if channels_first else data)\n        return wav, sr\n    torchaudio.load = _sf_load\n\n    def _sf_info(filepath, format=None, buffer_size=4096, backend=None):\n        import soundfile as sf\n        info = sf.info(str(filepath))\n        class _AudioMetaData:\n            pass\n        meta = _AudioMetaData()\n        meta.sample_rate = info.samplerate\n        meta.num_frames = info.frames\n        meta.num_channels = info.channels\n        meta.bits_per_sample = 16\n        meta.encoding = "PCM_S"\n        return meta\n    torchaudio.info = _sf_info\nexcept Exception:\n    pass\n'
    shim.write_text(SHIM_SRC, encoding="utf-8")

    # 書き出し事故（末尾のエスケープ崩れ等 → SyntaxError）を必ず py_compile で検証する
    r = subprocess.run([sys.executable,"-m","py_compile",str(shim)], capture_output=True, text=True)
    assert r.returncode == 0, "★shim が SyntaxError: " + r.stderr[-600:]

    pp = os.environ.get("PYTHONPATH","")
    if str(compat) not in pp.split(":"):
        os.environ["PYTHONPATH"] = f"{compat}:{pp}" if pp else str(compat)
    print("PYTHONPATH:", os.environ["PYTHONPATH"])

    # 機能確認: 別プロセスで torchaudio.load が shim（_compat）実装に置換されているか
    chk = subprocess.run([sys.executable,"-c",
        "import torchaudio;torchaudio.set_audio_backend('soundfile');"
        "import inspect;print('shim OK:', inspect.getsourcefile(torchaudio.load))"],
        capture_output=True, text=True, env=os.environ.copy())
    print(chk.stdout.strip() or chk.stderr[-800:])
    assert "shim OK" in chk.stdout and "_compat" in chk.stdout, "★shim が別プロセスに効いていない"


In [ ]:
# ===== §0-4 ローカル展開（fork コード / Zenodo データ / 学習成果物 / xvec_extract.py）=====
import os, shutil, subprocess
from pathlib import Path

ROOT = Path("/content/Style-Bert-VITS2")     # 評価実行ルート（ローカル）
DATA = ROOT / "Data" / "cv_r1"

# fork コードをローカルへ（Data / .git / model_assets / eval_out 除外 = 小ファイル地獄を回避）
!rsync -a --info=progress2 --exclude Data --exclude .git --exclude model_assets --exclude eval_out {DRIVE_BASE}/ {ROOT}/
os.chdir(ROOT); print("cwd:", Path.cwd())

# --- Zenodo からデータ取得（単一大ファイル DL → ローカル展開。Drive を経由しない）---
# Drive から小ファイル 8 万個を rsync/cp するのは同じ I/O 律速（数十 kB/s）で不可。
# 大ファイル 1 本の DL は桁違いに速い。VM ローカルディスクは 100 GB 以上空きがあり容量も問題ない。
DOI_REC = "https://zenodo.org/records/21119791/files"
META = "cadence_cv_r1_meta_v20260702.tgz"    # 31.9 MiB: config / esd / cadseq / MANIFEST 等
WAVS = "cadence_cv_r1_wavs_v20260702.tar"    # 4.81 GiB 無圧縮: wav 18015 (sr44100 mono)
DL = Path("/content/_zenodo"); DL.mkdir(exist_ok=True)

def n_files(d, pat): return sum(1 for _ in Path(d).rglob(pat)) if Path(d).exists() else 0

if (DATA/"config.json").exists() and n_files(DATA, "*.wav") == 18015:
    print("配置済み → DL / 展開をスキップ")
else:
    # --- 取得: Drive キャッシュ → aria2 16並列 → wget フォールバック ---------------
    # Zenodo は単一接続だと 1〜2 MB/s まで落ちることがある（4.8 GiB で 1 時間超）。
    # aria2 の並列レンジ DL で通常 5〜15 倍出る。DRIVE_BASE/zenodo_cache/ に tar を
    # 置いておけば Zenodo を経由せず Drive から複写する（単一大ファイルなので速い）。
    CACHE = DRIVE_BASE / "zenodo_cache"
    CACHE_TO_DRIVE = False      # True: DL 成功後に Drive へ保存（次回以降 Zenodo 不要。約 5 GB 消費）
    MIN_SIZE = {META: 30_000_000, WAVS: 5_000_000_000}   # 完全性の下限（途中切断の検出）
    def _ok(p, name): return p.exists() and p.stat().st_size >= MIN_SIZE[name]
    def _fetch(name):
        dst = DL / name
        if _ok(dst, name):
            print("取得済み:", name); return
        if dst.exists(): dst.unlink()   # 不完全ファイルは捨てる（低速 DL の再開より並列 DL のほうが速い）
        if _ok(CACHE / name, name):
            print("Drive キャッシュから複写:", name)
            shutil.copy2(CACHE / name, dst); return
        if not shutil.which("aria2c"):
            !apt-get -qq -y install aria2 > /dev/null
        url = f"{DOI_REC}/{name}?download=1"
        !aria2c -x16 -s16 -k1M --console-log-level=warn --summary-interval=15 -d {DL} -o {name} "{url}"
        if not _ok(dst, name):
            print("★aria2 失敗 → wget にフォールバック")
            !wget -c -O {DL}/{name} "{url}"
        assert _ok(dst, name), f"★DL 不完全: {name} (size={dst.stat().st_size if dst.exists() else 0})"
    for name in (META, WAVS):
        _fetch(name)
    if CACHE_TO_DRIVE:
        CACHE.mkdir(exist_ok=True)
        for name in (META, WAVS):
            if not _ok(CACHE / name, name):
                print("Drive へキャッシュ保存:", name); shutil.copy2(DL / name, CACHE / name)
    stage = Path("/content/_zenodo/x")
    if stage.exists(): shutil.rmtree(stage)
    stage.mkdir(parents=True)
    for name in (META, WAVS):
        print("展開:", name)
        !tar -xf {DL}/{name} -C {stage}
    # 展開先を自動判定: config.json を含むディレクトリ = データセットルート → Data/cv_r1 へ move（同一 FS 内で即時）
    cfgs = list(stage.rglob("config.json"))
    assert len(cfgs) == 1, f"★config.json の位置を特定できない: {cfgs}"
    src_root = cfgs[0].parent; print("データセットルート検出:", src_root)
    DATA.parent.mkdir(parents=True, exist_ok=True)
    if DATA.exists() and not DATA.is_symlink(): shutil.rmtree(DATA)
    shutil.move(str(src_root), str(DATA))
    if n_files(DATA, "*.wav") == 0:
        # wav tar のルート prefix が meta と異なる場合: 残りを DATA 直下へ統合
        for child in list(stage.iterdir()):
            print("統合:", child.name, "->", DATA/child.name)
            shutil.move(str(child), str(DATA/child.name))
print("wav:", n_files(DATA, "*.wav"), "(18015 が正) / cadseq:", n_files(DATA, "*.cadseq.npy"), "(11010 が正)")

# 学習成果物: model_assets は Drive の実体へ symlink（学習ノートが書き出した ckpt / style_vectors.npy を参照）
drive_assets = DRIVE_BASE / "model_assets"
assert drive_assets.exists(), "★Drive に model_assets が無い（学習の完走を確認: cv_r1_train_colab.ipynb §8）"
local_assets = ROOT / "model_assets"
if local_assets.exists() and not local_assets.is_symlink(): shutil.rmtree(local_assets)
!ln -sfn {drive_assets} {local_assets}
print("model_assets ->", os.path.realpath(local_assets))

# xvec_extract.py を fork 直下へ配置（x-vector 抽出は TF/CPU の別プロセスとして呼び出すため）
src = ROOT / "colab" / "xvec_extract.py"
if src.exists() and not (ROOT / "xvec_extract.py").exists():
    shutil.copy(src, ROOT / "xvec_extract.py")
if not (ROOT / "xvec_extract.py").exists():
    # Drive clone が古く colab/ に未反映の場合 → fork の raw から直接取得
    print("colab/ に xvec_extract.py が無い → GitHub から取得（Drive clone が古い。次回 git pull か setup 再実行を推奨）")
    RAW = "https://raw.githubusercontent.com/slp-hu/Style-Bert-VITS2/layer-b-cadence-seq/colab/xvec_extract.py"
    !wget -q -O {ROOT}/xvec_extract.py "{RAW}"
xp = ROOT / "xvec_extract.py"
assert xp.exists() and xp.stat().st_size > 1000 and "def " in xp.read_text(encoding="utf-8"), \
    "★xvec_extract.py を取得できない（fork の colab/ に commit 済みか確認。または手動で ROOT 直下にアップロード）"
print("§0-4 完了")


In [ ]:
# ===== §1 設定・モデルロード（07a 合成コア準拠）=====
import os, sys, re
BASE = "/content/Style-Bert-VITS2"
os.chdir(BASE); sys.path.insert(0, BASE)
from pathlib import Path
import numpy as np, torch

DEVICE  = "cuda" if torch.cuda.is_available() else "cpu"
DATA    = f"{BASE}/Data/cv_r1"
CONFIG  = f"{DATA}/config.json"

# 学習済み ckpt を自動検出（総 step はバケットサンプラの丸めで 8,910〜9,019 程度に揺れるため
# ファイル名を固定しない。最大 step の safetensors を採用。固定したい場合は CKPT を直接上書き）
_all = list(Path(BASE, "model_assets").glob("*/*_e*_s*.safetensors"))
_c = sorted((p for p in _all if p.parent.name != "cv_r1_smoke"),   # スモークの成果物は評価対象外
            key=lambda p: int(re.search(r"_s(\d+)\.safetensors$", p.name).group(1)))
assert _c, ("★本番学習の safetensors が model_assets に無い（学習の完走と §0-4 の symlink を確認）"
            + ("。cv_r1_smoke のみ検出 = スモーク学習しか回していない。スモークは動作確認用で評価に使えない"
               if _all else ""))
CKPT    = str(_c[-1])
SV_PATH = str(Path(CKPT).parent / "style_vectors.npy")
print("CKPT:", CKPT)
assert Path(CONFIG).exists(),  f"config 無し: {CONFIG}（§0-4 のデータ展開を確認）"
assert Path(SV_PATH).exists(), f"style_vectors 無し: {SV_PATH}"

from style_bert_vits2.tts_model import TTSModel
from style_bert_vits2.nlp import bert_models
from style_bert_vits2.constants import Languages
bert_models.load_model(Languages.JP, "ku-nlp/deberta-v2-large-japanese-char-wwm")
bert_models.load_tokenizer(Languages.JP, "ku-nlp/deberta-v2-large-japanese-char-wwm")

model = TTSModel(model_path=Path(CKPT), config_path=Path(CONFIG),
                 style_vec_path=Path(SV_PATH), device=DEVICE)
model.load(); net_g = model.net_g; net_g.eval()
hps = model.hyper_parameters
NEUTRAL = model.get_style_vector(0, 1.0)
spk2id = hps.data.spk2id
SR = hps.data.sampling_rate

# cadence_cond 学習済みサニティ
sd = net_g.state_dict()
cad = {k: float(sd[k].float().norm()) for k in sd if "cadence_cond" in k}
print("cadence_cond ‖w‖:", {k: round(v,4) for k,v in cad.items()})
assert any("weight" in k and v > 1e-6 for k,v in cad.items()), "cadence_cond 未学習の疑い"
print("loaded | n_spk:", len(spk2id), "| SR:", SR, "| device:", DEVICE)


In [ ]:
# ===== §2 合成コア（WEIGHT=0.0 既定：weight==0 は cadence_vec=None＝非注入）=====
from style_bert_vits2.models.infer import get_text
CADENCE_DIM = 32
WEIGHT = 0.0   # 0.0=cadence 非注入（emb_g だけで弾き分け）。>0 で self cadseq を注入。

def parse_esd(path):
    recs=[]
    for line in open(path, encoding="utf-8"):
        line=line.rstrip("\n")
        if not line: continue
        wav, sid, lang, text, phones, tone, word2ph = line.split("|")
        recs.append(dict(wav=wav, spk=str(sid), text=text,
                         phones=phones.split(" "), tones=[int(t) for t in tone.split(" ")]))
    return recs

def cadseq_to_vec(cadseq, T_x, add_blank=True):
    cadt = torch.FloatTensor(cadseq)
    if add_blank:
        b = torch.zeros(cadt.size(0)*2+1, cadt.size(1)); b[1::2]=cadt; cadt=b
    assert cadt.shape == (T_x, CADENCE_DIM), f"cadence整列ミスマッチ {tuple(cadt.shape)} vs ({T_x},{CADENCE_DIM})"
    return cadt.transpose(0,1).contiguous().unsqueeze(0)

def load_self_cadseq(wav):
    p = os.path.join(BASE, wav) + ".cadseq.npy" if not wav.startswith("/") else wav+".cadseq.npy"
    return np.load(p).astype(np.float32) if os.path.exists(p) else None

@torch.no_grad()
def synth(text, phones, tones, spk_name, weight=WEIGHT, self_wav=None,
          sdp_ratio=0.0, noise_scale=0.667, noise_scale_w=0.8, length_scale=1.0):
    net_g.cadence_weight_dp  = float(weight)
    net_g.cadence_weight_sdp = float(weight)
    bert, ja_bert, en_bert, phone, tone, lang = get_text(
        text, Languages.JP, hps, DEVICE, given_phone=phones, given_tone=tones)
    T_x = phone.size(0)
    # weight==0 → cadence 非注入（None）。weight>0 → self cadseq（無ければ zeros）を注入。
    cad_t = None
    if weight != 0.0:
        cs = load_self_cadseq(self_wav) if self_wav else None
        if cs is None:
            P = (T_x - 1)//2 if hps.data.add_blank else T_x
            cs = np.zeros((P, CADENCE_DIM), np.float32)
        cad_t = cadseq_to_vec(cs, T_x, hps.data.add_blank).to(DEVICE)
    x      = phone.to(DEVICE).unsqueeze(0)
    x_len  = torch.LongTensor([phone.size(0)]).to(DEVICE)
    sid_t  = torch.LongTensor([spk2id[spk_name]]).to(DEVICE)
    tone_t = tone.to(DEVICE).unsqueeze(0)
    lang_t = lang.to(DEVICE).unsqueeze(0)
    jb_t   = ja_bert.to(DEVICE).unsqueeze(0)
    style_t= torch.from_numpy(NEUTRAL).to(DEVICE).unsqueeze(0)
    out = net_g.infer(x, x_len, sid_t, tone_t, lang_t, jb_t,
                      style_vec=style_t, cadence_vec=cad_t,
                      sdp_ratio=sdp_ratio, noise_scale=noise_scale,
                      noise_scale_w=noise_scale_w, length_scale=length_scale)
    return out[0][0,0].data.cpu().float().numpy()

print("合成コア OK / WEIGHT =", WEIGHT)


In [ ]:
# ===== §3 弾き分けデータ生成（先頭 N=10 話者 × M=3 発話、原音と同一発話を合成）=====
import soundfile as sf, collections
N_SPK, M_UTT = 10, 3
OUT = f"{BASE}/eval_out"; os.makedirs(f"{OUT}/synth", exist_ok=True)

recs = parse_esd(f"{DATA}/esd_train.list")
by_spk = collections.defaultdict(list)
for r in recs: by_spk[r["spk"]].append(r)

target_spks = [f"cv_{i:04d}" for i in range(1, N_SPK+1)]
items = {}   # key -> abs wav path（real と synth の両方）
meta  = []   # (spk, uid, key_real, key_synth)
for spk in target_spks:
    assert spk in by_spk, f"{spk} が esd に無い"
    for j, r in enumerate(by_spk[spk][:M_UTT]):
        uid = f"u{j}"
        real_abs = os.path.join(BASE, r["wav"])
        wav = synth(r["text"], r["phones"], r["tones"], spk, weight=WEIGHT, self_wav=r["wav"])
        synth_path = f"{OUT}/synth/{spk}_{uid}.wav"
        sf.write(synth_path, wav, SR)
        kr, ks = f"real__{spk}__{uid}", f"synth__{spk}__{uid}"
        items[kr] = real_abs; items[ks] = synth_path
        meta.append((spk, uid, kr, ks))
    print("done:", spk)

import json
json.dump(items, open(f"{OUT}/items.json","w"), ensure_ascii=False, indent=1)
json.dump(meta,  open(f"{OUT}/meta.json","w"),  ensure_ascii=False)
print(f"合成 {N_SPK*M_UTT} + 原音 {N_SPK*M_UTT} = {len(items)} 件 / items.json 出力")


In [ ]:
# ===== §4 x-vector 抽出（TF/CPU 別プロセス。合成側の torch/GPU と分離）=====
# xvec_extract.py は fork 直下に配置済みの前提。初回は pip 導入が走る。
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "vad", "tensorflow_io", "librosa", "soundfile", "tensorflow"], check=False)
r = subprocess.run([sys.executable, "xvec_extract.py",
                    "--items", f"{OUT}/items.json", "--out", f"{OUT}/xvec.npz",
                    "--drive", "/content/drive/MyDrive"])
assert r.returncode == 0, "xvec_extract 失敗（ログ確認）"
xv = np.load(f"{OUT}/xvec.npz")
print("x-vector keys:", len(xv.files), "/ dim:", xv[xv.files[0]].shape)


In [ ]:
# ===== §5 弾き分け指標（cos 行列 / top-1 識別率 / 話者内・間分布）=====
import numpy as np, json
import matplotlib.pyplot as plt
meta = json.load(open(f"{OUT}/meta.json"))
xv = np.load(f"{OUT}/xvec.npz")
def vget(k): 
    v = xv[k]; return v/(np.linalg.norm(v)+1e-9)

spks = [f"cv_{i:04d}" for i in range(1, N_SPK+1)]
# 原音の話者平均を「登録話者テンプレート」に
real_by_spk = collections.defaultdict(list)
for (spk,uid,kr,ks) in meta: real_by_spk[spk].append(vget(kr))
real_mean = {s: np.mean(real_by_spk[s],0) for s in spks}
Rmat = np.stack([real_mean[s]/(np.linalg.norm(real_mean[s])+1e-9) for s in spks])  # (N,150)

# 各合成発話 → 最近傍原音話者（top-1 識別率）
correct, total = 0, 0
S_syn = collections.defaultdict(list)
for (spk,uid,kr,ks) in meta:
    sv = vget(ks); sims = Rmat @ sv
    pred = spks[int(np.argmax(sims))]
    correct += (pred==spk); total += 1
    S_syn[spk].append(sv)
acc = correct/total
print(f"top-1 話者識別率（合成→原音テンプレート）: {acc*100:.1f}%  (chance {100/N_SPK:.1f}%, {correct}/{total})")

# 合成話者平均 × 原音話者平均 の cos 行列
syn_mean = {s: np.mean(S_syn[s],0) for s in spks}
Smat = np.stack([syn_mean[s]/(np.linalg.norm(syn_mean[s])+1e-9) for s in spks])
C = Smat @ Rmat.T   # (N,N) synth×real
fig, ax = plt.subplots(figsize=(6,5))
im = ax.imshow(C, cmap="viridis"); fig.colorbar(im)
ax.set_xticks(range(N_SPK)); ax.set_xticklabels(spks, rotation=90, fontsize=7)
ax.set_yticks(range(N_SPK)); ax.set_yticklabels(spks, fontsize=7)
ax.set_xlabel("real (template)"); ax.set_ylabel("synth"); ax.set_title(f"cos sim  (diag mean={np.diag(C).mean():.3f})")
plt.tight_layout(); plt.savefig(f"{OUT}/simmat.png", dpi=120); plt.show()

# 話者内 vs 話者間（合成→原音テンプレート）
within = [ (Smat[i]@Rmat[i]) for i in range(N_SPK) ]
between = [ (Smat[i]@Rmat[j]) for i in range(N_SPK) for j in range(N_SPK) if i!=j ]
print(f"cos 話者内 mean={np.mean(within):.3f} / 話者間 mean={np.mean(between):.3f} / 差={np.mean(within)-np.mean(between):.3f}")


In [ ]:
# ===== §6 自然性 UTMOS（合成 vs 原音、torch.hub。CPU フォールバック）=====
import torch, numpy as np, soundfile as sf, librosa, json
try:
    _dev = "cuda" if torch.cuda.is_available() else "cpu"
    utmos = torch.hub.load("tarepan/SpeechMOS:v1.2.0", "utmos22_strong", trust_repo=True).to(_dev).eval()
    def utmos_score(path):
        y, sr = sf.read(path, dtype="float32")
        if getattr(y,"ndim",1) > 1: y = y.mean(1)
        if sr != 16000: y = librosa.resample(y, orig_sr=sr, target_sr=16000)
        with torch.no_grad():
            t = torch.from_numpy(y).float().unsqueeze(0).to(_dev)
            return float(utmos(t, 16000).cpu().item())
    meta = json.load(open(f"{OUT}/meta.json")); items = json.load(open(f"{OUT}/items.json"))
    real_s = [utmos_score(items[kr]) for (_,_,kr,ks) in meta]
    syn_s  = [utmos_score(items[ks]) for (_,_,kr,ks) in meta]
    print(f"UTMOS  原音 mean={np.mean(real_s):.3f}±{np.std(real_s):.3f} / 合成 mean={np.mean(syn_s):.3f}±{np.std(syn_s):.3f}")
    print(f"       自然性ギャップ（原音-合成）= {np.mean(real_s)-np.mean(syn_s):.3f}")
    json.dump({"real":real_s,"synth":syn_s}, open(f"{OUT}/utmos.json","w"))
except Exception as e:
    print("UTMOS スキップ（要確認）:", repr(e)[:200])


In [ ]:
# ===== §7 virtual 話者（emb_g 補間 α=0..1 / 複数話者平均）＋試聴 =====
from IPython.display import display, Audio, Markdown

@torch.no_grad()
def synth_with_embg(text, phones, tones, ref_spk, g_vec, weight=0.0):
    # emb_g[slot] を g_vec に一時差し替えて合成（ref_spk の slot を借用）→ 復元
    emb_w = net_g.emb_g.weight.data
    slot = spk2id[ref_spk]; backup = emb_w[slot].clone()
    emb_w[slot] = g_vec.to(emb_w.device, emb_w.dtype)
    try:
        return synth(text, phones, tones, ref_spk, weight=weight)
    finally:
        emb_w[slot] = backup

emb = net_g.emb_g.weight.data
def gvec(spk): return emb[spk2id[spk]].clone()

# 試聴用の1発話（cv_0001 の先頭発話テキストを使う）
r0 = by_spk["cv_0001"][0]
TXT, PH, TN = r0["text"], r0["phones"], r0["tones"]

display(Markdown("### 補間 cv_0001 ↔ cv_0002 (α: 0→1)"))
A, B = gvec("cv_0001"), gvec("cv_0002")
for a in [0.0, 0.25, 0.5, 0.75, 1.0]:
    g = (1-a)*A + a*B
    w = synth_with_embg(TXT, PH, TN, "cv_0001", g, weight=0.0)
    display(Markdown(f"**α={a:.2f}**")); display(Audio(w, rate=SR, normalize=True))

display(Markdown("### 平均話者（cv_0001..cv_0010 の重心＝誰でもない声）"))
gmean = torch.stack([gvec(f"cv_{i:04d}") for i in range(1,11)]).mean(0)
w = synth_with_embg(TXT, PH, TN, "cv_0001", gmean, weight=0.0)
display(Audio(w, rate=SR, normalize=True))
print("virtual 話者 試聴 OK")


In [ ]:
# ===== §8 成果物を Drive へ退避 =====
import shutil, os
DST = "/content/drive/MyDrive/Style-Bert-VITS2/eval_out"
os.makedirs(DST, exist_ok=True)
for f in ["items.json","meta.json","xvec.npz","simmat.png","utmos.json"]:
    p = f"{OUT}/{f}"
    if os.path.exists(p): shutil.copy(p, f"{DST}/{f}"); print("saved:", f)
# 合成 wav はサイズ次第。数十件なら退避
shutil.make_archive(f"{DST}/synth_wavs", "zip", f"{OUT}/synth")
print("saved: synth_wavs.zip ->", DST)


## §9（任意・1回だけ）デモ用アセットの生成

合成ノート（§2.5）とデモ（demo/app.py）が使う **参照クリップ**（各話者の元音声）と
**話者マップ**（x-vector の UMAP 2次元座標）を生成する。出力先はどちらも
`model_assets/` 直下の共有置き場（Drive 永続・モデル非依存）なので、一度生成すれば
以後どのセッション・どのモデル（本番/スモーク）からも自動で使われる。
評価（§1〜§8）とは独立で、**§0 完了直後でも実行できる**。


In [ ]:
# ===== §9 参照クリップ + 話者マップの生成（任意・1回だけ / 冪等）=====
FORCE = False   # True で再生成
import subprocess, sys
from pathlib import Path

# --- 参照クリップ（各話者 1 発話、Opus 圧縮、全 298 話者で 10〜20 MB。追加 pip 不要）---
_clips = Path("model_assets/reference_clips/reference_clips.json")
if _clips.exists() and not FORCE:
    print("参照クリップ: 生成済み → スキップ")
else:
    !python demo/make_reference_clips.py --data Data/cv_r1 --out model_assets/reference_clips

# --- 話者マップ（x-vector 平均 → UMAP。TF/CPU の x-vector 抽出は数分）---
_map = Path("model_assets/speaker_map.json")
if _map.exists() and not FORCE:
    print("話者マップ: 生成済み → スキップ")
else:
    # ★umap-learn を無指定で入れると依存解決が numpy を 2.x へ動かし、環境が壊れる。
    #   numpy/scipy の pin を同時指定して resolver に動かさせない。
    !pip install -q umap-learn "numpy==1.26.4" "scipy==1.13.1"
    # 保険: それでも numpy が動かされていたら敷き直す（別プロセスで健全性確認）
    _h = subprocess.run([sys.executable, "-c", "import numpy.random, numpy; print(numpy.__version__)"],
                        capture_output=True, text=True)
    if _h.returncode != 0:
        print("★umap 導入で numpy が壊れた → 1.26.4 を強制再インストールで修復")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--force-reinstall",
                        "--no-deps", "--no-cache-dir", "numpy==1.26.4"], check=True)
        _h = subprocess.run([sys.executable, "-c", "import numpy.random, numpy; print(numpy.__version__)"],
                            capture_output=True, text=True)
        assert _h.returncode == 0, "★numpy を修復できない:\n" + (_h.stderr or "")[-600:]
    print("numpy 健全性 OK:", _h.stdout.strip())
    !python demo/make_speaker_map.py --data Data/cv_r1 --out model_assets/speaker_map.json

import json as _json
if _clips.exists():
    print("clips:", len(_json.load(open(_clips, encoding="utf-8"))), "話者")
if _map.exists():
    print("map  :", len(_json.load(open(_map, encoding="utf-8"))), "話者")
print("→ synth ノート §2.5 / demo が次回から自動で使う")
